In [1]:
# TASK 1: Import the LLM wrapper class from langchain_ollama
# Hint: You need a class that wraps a local Ollama model.
# Classes in LangChain that wrap LLMs often end in "LLM".
from langchain_ollama import OllamaLLM

# TASK 2: Instantiate the LLM pointing at your local llama3.2 model
# Hint: The parameter that specifies which model to use is called "model"
llm = OllamaLLM(model="llama3.2")

In [2]:
# TASK 3: Create a mock_context string containing ALL of:
#   - A fake calendar for today (at least 3 time-blocked events)
#   - A list of 2-3 pending tasks with priority levels
#   - The question at the end: "What should I focus on today?"
mock_context = """
Today's Calendar:
- 9:00 AM  : Team standup (15 min)
- 11:00 AM : Code review — PR #142 auth refactor
- 2:00 PM  : 1:1 with manager
- 4:00 PM  : Free block (no meetings)

Pending Tasks:
- Fix auth bug in login flow (HIGH priority — due TODAY, blocking 2 teammates)
- Write unit tests for payment module (MEDIUM — due end of week)
- Review Jira backlog for next sprint (LOW — no deadline)

Question: What should I focus on today?
"""

# TASK 4: Print a confirmation message so you know Cell 1 succeeded
print("setup complete")


setup complete


In [3]:
# ═══════════════════════════════════════
# STAGE 1: THINK
# Purpose: Get an initial answer from the LLM
# ═══════════════════════════════════════

# TASK 1: Write a prompt that:
#   - Gives the LLM a persona ("You are a productivity assistant...")
#   - Injects mock_context using an f-string
#   - Asks what the user should focus on today
initial_prompt = f"""
You are a personal productivity assistant.
Given the following context about a developer's day, suggest what they should
focus on today. Be concise and practical — output a short prioritised list.
{mock_context}
"""
initial_answer= llm.invoke(initial_prompt)
print("=== INITIAL ANSWER ===")
print(initial_answer)
print()

=== INITIAL ANSWER ===
Based on the context, here is a prioritized list of tasks to focus on today:

1. **Fix auth bug in login flow** (HIGH priority - due TODAY, blocking 2 teammates)
2. **Code review - PR #142 auth refactor** (next priority, ongoing)
3. **Free block (4:00 PM)** (use this time for a task of your choice, e.g., making progress on unit tests for payment module)

I recommend focusing on the high-priority task first, as it's blocking 2 teammates and needs to be completed today. After completing that task, you can allocate the free block to a less pressing task, like making progress on the payment module unit tests.



In [4]:
# Stage 2 : Reflection
reflection_prompt =f"""
You are a critical productivity reviewer.
Examine the following suggestion and evaluate it honestly:
1. What is good about this recommendation?
2. What is missing or could be improved?
3. Is the prioritisation logical given blocking dependencies?
4. Is anything overlooked?

Suggestion to review:
{initial_answer}

"""
critique = llm.invoke(reflection_prompt)
print("=== CRITIQUE ===")
print(critique)
print()

=== CRITIQUE ===
**Evaluation of the Suggestion**

**1. What is good about this recommendation?**

The recommendation shows a good understanding of prioritization and blocking dependencies. The high-priority task, "Fix auth bug in login flow", is clearly identified as blocking 2 teammates, which indicates a high impact on the team's productivity and collaboration. The recommendation also includes a clear next priority, "Code review - PR #142 auth refactor", which suggests a structured approach to task management.

**2. What is missing or could be improved?**

One area for improvement is the lack of details about the "free block" task. While the suggestion mentions allocating this time to making progress on unit tests for the payment module, it would be beneficial to include more context, such as:
	* Specific goals or objectives for the payment module unit tests
	* Estimated time required for the task
	* Potential dependencies or blocking factors

Additionally, the recommendation could 

In [5]:
# stage 3 Refine
refinement_prompt =f"""
You are a personal productivity assistant.
You previously gave this recommendation:
{initial_answer}

A critical reviewer gave this feedback:
{critique}

Using the critique, produce an improved final recommendation.
Be concise, specific, and actionable. Output a clear prioritised focus list.
"""
final_answer = llm.invoke(refinement_prompt)
print("=== FINAL ANSWER ===")
print(final_answer)

=== FINAL ANSWER ===
**Improved Final Recommendation**

Based on the context and feedback, here is an updated prioritized list of tasks to focus on today:

1. **Fix auth bug in login flow** (HIGH priority - due TODAY, blocking 2 teammates)
	* Estimated time required: 2 hours
	* Specific goals: Complete the fix and ensure the login flow is stable
	* Dependencies: None
2. **Code review - PR #142 auth refactor** (NEXT priority, ONGOING)
	* Estimated time required: 1.5 hours
	* Specific goals: Review the code changes and ensure they meet the required standards
	* Dependencies: None
3. **Unit tests for payment module (PaymentModuleUT)** (MEDIUM priority, FREE BLOCK)
	* Estimated time required: 2 hours
	* Specific goals: Complete the missing unit tests for the payment module
	* Dependencies: None

**Additional Considerations**

* Schedule a meeting with the 2 teammates who are blocked by the auth bug to discuss the status and estimated time required to complete the fix.
* Reach out to the co

In [6]:
# package as resuable agent
def reflection_agent(user_context: str) -> dict:
    """
    Reflection Loop Agent.

    Implements: Think -> Reflect -> Refine
    Accepts any context string, returns all reasoning stages.

    Args:
        user_context: String containing calendar, tasks, and a question.

    Returns:
        dict with keys: initial_answer, critique, final_answer, reasoning_trace

    Note:
        reasoning_trace is currently a list of stage names.
        In Week 9, this becomes a structured LangSmith trace with
        timestamps, token counts, and model metadata per stage.
    """

    # Stage 1: Think
    think = llm.invoke(
        f"You are a productivity assistant. "
        f"What should the user focus on today? Be concise.\n\n{user_context}"
    )

    # Stage 2: Reflect
    reflect = llm.invoke(
        f"You are a critical reviewer. Evaluate this productivity suggestion. "
        f"What is good, missing, or illogical?\n\n{think}"
    )

    # Stage 3: Refine
    refine = llm.invoke(
        f"You are a productivity assistant. "
        f"Improve this recommendation using the critique. Be concise and actionable.\n\n"
        f"Original: {think}\n\nCritique: {reflect}"
    )
    
    # stage 4 summary 
    summary = llm.invoke (
        f"Summarise this recommendation in a single sentence (max 20 words):\n\n{refine}"
    )

    return {
        "initial_answer"  : think,
        "critique"        : reflect,
        "final_answer"    : refine,
        "summary"         : summary, 
        "reasoning_trace" : ["think", "reflect", "refine","summary"]
    }

In [7]:
# Run it with the mock context
result = reflection_agent(mock_context)

print("=" * 50)
print("AGENT RUN COMPLETE")
print("=" * 50)
print("\nFinal Answer:\n", result["final_answer"])
print("\nReasoning Trace:", result["reasoning_trace"])


AGENT RUN COMPLETE

Final Answer:
 Here's an improved recommendation:

"Given your priorities and schedule, I recommend focusing on fixing the authentication bug in the login flow today, as it has a HIGH priority and is blocking 2 teammates. To ensure you can complete this task without impacting your own well-being, please take a 10-minute break after lunch to recharge.

Additionally, I suggest breaking this task into smaller, manageable chunks, working with your colleague on the task, and prioritizing the most critical bug fixes first. If you're unable to complete the task today, consider working with our team lead to explore alternative solutions.

Please review the team's overall workload and adjust your schedule as needed to ensure you can complete this task without impacting your own energy levels or other tasks. Our team's priority is to ensure the login flow is stable and secure, so your efforts will directly impact the team's success."

This revised recommendation:

1. Adds a c

In [8]:
##### END #####
print("##### END #####")

##### END #####
